# Zomato Restaurant Performance Analysis

## 1. Introduction

This notebook provides a comprehensive analysis of a Zomato restaurant dataset. The goal is to identify factors contributing to restaurant performance, define and predict 'underperforming' establishments, and derive insights that can help in understanding restaurant dynamics, pricing strategies, and potential areas for business improvement.

We will cover the following steps:
1.  **Data Loading and Initial Inspection**
2.  **Data Cleaning and Preprocessing**
3.  **Feature Engineering**
4.  **Exploratory Data Analysis (EDA)**
5.  **Defining and Analyzing 'Underperforming' Restaurants**
6.  **Predictive Modeling**
7.  **Exporting Analysis Results**
8.  **Full Report Summary**

## 2. Data Loading and Initial Inspection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os

dataset_path = '/content/zomato_dataset.csv'

try:
    zomato_df = pd.read_csv(dataset_path)
    print(f"Dataset '{dataset_path}' loaded successfully. Here are the first 5 rows:")
    display(zomato_df.head())
    print("\nAnd here is some basic information about the dataset:")
    zomato_df.info()
except FileNotFoundError:
    print(f"Error: The dataset '{dataset_path}' was not found.")
    print("Please ensure you have uploaded 'zomato.csv' to your Colab environment or provide the correct path/URL.")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

## 3. Data Cleaning and Preprocessing

### Handling Missing Values

First, let's identify how many missing values each column has. Then, we'll decide on a strategy to handle them. For `Dining_Rating` and `Delivery_Rating`, we'll fill `NaN` values with `0`, assuming that a missing rating implies it hasn't been rated or doesn't apply. For `Best_Seller`, which is a categorical column, we'll fill `NaN` with 'Not Available'.

In [ ]:
# Check for missing values before handling
print("Missing values before cleaning:")
print(zomato_df.isnull().sum())

# Fill missing values in 'Dining_Rating' and 'Delivery_Rating' with 0
zomato_df['Dining_Rating'] = zomato_df['Dining_Rating'].fillna(0)
zomato_df['Delivery_Rating'] = zomato_df['Delivery_Rating'].fillna(0)

# Fill missing values in 'Best_Seller' with 'Not Available'
zomato_df['Best_Seller'] = zomato_df['Best_Seller'].fillna('Not Available')

# Verify missing values after cleaning
print("\nMissing values after cleaning:")
print(zomato_df.isnull().sum())

### Checking for Duplicate Rows

Duplicate rows can skew analyses and models. Let's check if there are any exact duplicate rows in our dataset and remove them if found.

In [ ]:
# Check for duplicate rows
duplicate_rows_count = zomato_df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_rows_count}")

if duplicate_rows_count > 0:
    print("Removing duplicate rows...")
    zomato_df.drop_duplicates(inplace=True)
    print(f"Number of rows after removing duplicates: {len(zomato_df)}")
else:
    print("No duplicate rows found.")

# Display the first few rows and info of the cleaned DataFrame
print("\nFirst 5 rows of the cleaned DataFrame:")
display(zomato_df.head())
print("\nInfo of the cleaned DataFrame:")
zomato_df.info()

## 4. Feature Engineering

We will create new features that might be useful for our analysis and predictive modeling, such as 'Price per Vote', 'Popularity', and 'Rating Difference'.

In [ ]:
# Price per vote (value perception)
zomato_df['Price_per_Vote'] = zomato_df['Prices'] / (zomato_df['Votes'] + 1)

# Popularity score
zomato_df['Popularity'] = zomato_df['Dining Votes'] + zomato_df['Delivery_Votes']

# Rating difference
zomato_df['Rating_Diff'] = zomato_df['Dining_Rating'] - zomato_df['Delivery_Rating']

print("New features 'Price_per_Vote', 'Popularity', and 'Rating_Diff' have been added to zomato_df.")
print("Displaying the first 5 rows with the new features:")
display(zomato_df[['Restaurant_Name', 'Prices', 'Votes', 'Dining_Rating', 'Delivery_Rating', 'Price_per_Vote', 'Popularity', 'Rating_Diff']].head())

## 5. Exploratory Data Analysis (EDA)

We will now explore the distributions of key variables and relationships between them.

### Distribution of 'Dining_Rating'

Let's visualize the distribution of the `Dining_Rating` column using a histogram. This will show us the frequency of different rating values.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(zomato_df['Dining_Rating'], bins=20, kde=True)
plt.title('Distribution of Dining Rating')
plt.xlabel('Dining Rating')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

### Distribution of 'Delivery_Rating'

Now, let's visualize the distribution of the `Delivery_Rating` column. This will provide insights into how delivery ratings are spread.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(zomato_df['Delivery_Rating'], bins=20, kde=True)
plt.title('Distribution of Delivery Rating')
plt.xlabel('Delivery Rating')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

### Distribution of 'Prices'

Let's visualize the distribution of the 'Prices' column using a histogram. This will give us an idea of the price ranges in the dataset.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(zomato_df['Prices'], bins=30, kde=True)
plt.title('Distribution of Prices')
plt.xlabel('Prices')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

### Scatter Plot of 'Dining_Rating' vs. 'Prices'

Let's create a scatter plot to explore the relationship between `Dining_Rating` and `Prices`. This can help us understand if higher-rated restaurants tend to have higher or lower prices.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Dining_Rating', y='Prices', data=zomato_df, alpha=0.6)
plt.title('Dining Rating vs. Prices')
plt.xlabel('Dining Rating')
plt.ylabel('Prices')
plt.grid(True, alpha=0.75)
plt.show()

### Average Dining Rating by City

Let's calculate the average `Dining_Rating` for each city to see which cities have higher or lower average ratings.

In [ ]:
average_dining_rating_by_city = zomato_df.groupby('City')['Dining_Rating'].mean().reset_index()

print("Average Dining Rating by City:")
display(average_dining_rating_by_city.sort_values(by='Dining_Rating', ascending=False))

### Top 10 Cities by Average Dining Rating

To better visualize which cities have the highest average `Dining_Rating`, let's plot the top 10 cities in a bar chart.

In [ ]:
# Get the top 10 cities by average dining rating
top_10_cities_dining = average_dining_rating_by_city.head(10)

plt.figure(figsize=(12, 7))
sns.barplot(x='City', y='Dining_Rating', data=top_10_cities_dining, palette='viridis')
plt.title('Top 10 Cities by Average Dining Rating')
plt.xlabel('City')
plt.ylabel('Average Dining Rating')
plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()

### Top 10 Restaurants by Average Dining Rating

Let's find the top 10 restaurants based on their average `Dining_Rating`. We will exclude restaurants where the `Dining_Rating` was imputed as 0, as this likely means they haven't been rated for dining.

In [ ]:
rated_restaurants_df = zomato_df[zomato_df['Dining_Rating'] > 0]

# Calculate the average Dining_Rating for each restaurant
average_restaurant_dining_rating = rated_restaurants_df.groupby('Restaurant_Name')['Dining_Rating'].mean().reset_index()

# Sort by Dining_Rating in descending order and get the top 10
top_10_restaurants = average_restaurant_dining_rating.sort_values(by='Dining_Rating', ascending=False).head(10)

print("Top 10 Restaurants by Average Dining Rating:")
display(top_10_restaurants)

### Visualizing Top 10 Restaurants by Average Dining Rating

To make the comparison clearer, let's visualize these top 10 restaurants using a bar chart.

In [ ]:
plt.figure(figsize=(14, 8))
sns.barplot(x='Dining_Rating', y='Restaurant_Name', data=top_10_restaurants, palette='coolwarm', hue='Restaurant_Name', legend=False)
plt.title('Top 10 Restaurants by Average Dining Rating')
plt.xlabel('Average Dining Rating')
plt.ylabel('Restaurant Name')
plt.grid(axis='x', alpha=0.75)
plt.tight_layout()
plt.show()

### Scatter Plot of 'Delivery_Rating' vs. 'Prices'

Now, let's explore the relationship between `Delivery_Rating` and `Prices` using a scatter plot. This will help us understand if restaurants with higher delivery ratings tend to have different pricing strategies.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Delivery_Rating', y='Prices', data=zomato_df, alpha=0.6)
plt.title('Delivery Rating vs. Prices')
plt.xlabel('Delivery Rating')
plt.ylabel('Prices')
plt.grid(True, alpha=0.75)
plt.show()

## Installing Python Libraries

In Google Colab, you can install Python libraries using `pip` by prefixing the command with an exclamation mark `!`. This tells Colab to run the command in the shell.

In [ ]:
# To install a library, replace 'my_library' with the actual library name
!pip install my_library

# For example, to install the pandas library:
# !pip install pandas

# You can also install specific versions or from a requirements.txt file:
# !pip install pandas==1.3.5
# !pip install -r requirements.txt

After installation, you can import and use the library in your Python code.

In [ ]:
# Import the library
import my_library

# Or, to import with an alias (a shorter name):
# import my_library as ml

Once imported, you can use the functions and classes provided by the library. For example, if `my_library` had a function called `do_something()`, you would call it like this:

```python
my_library.do_something()
```

If you imported it with an alias, like `import my_library as ml`, you would use:

```python
ml.do_something()
```

### Using `dir()` to list contents

The `dir()` function, when called on an imported module, returns a list of names (attributes, functions, classes, etc.) defined in that module.

In [ ]:
# List all attributes and functions in 'my_library'
print(dir(my_library))

### Using `help()` to get detailed information

The `help()` function provides more detailed documentation (docstrings) for a module, class, function, or method. It's very useful for understanding what each component does and how to use it.

In [ ]:
# Get detailed help for 'my_library'
help(my_library)

# If you find a specific function using dir(), you can get help on it:
# For example, if my_library had a function 'do_something':
# help(my_library.do_something)

In [ ]:
# To start with data cleaning, we typically use the pandas library.
# First, let's make sure it's installed (it usually is in Colab).
# !pip install pandas

import pandas as pd

# Now, we can load a sample dataset. Colab comes with some sample data.
# For example, let's load a CSV file from the sample_data directory.
# This is a common first step in data cleaning.

try:
    df = pd.read_csv('/content/sample_data/california_housing_train.csv')
    print("Dataset loaded successfully. Here are the first 5 rows:")
    print(df.head())
    print("\nAnd here is some basic information about the dataset:")
    df.info()
except FileNotFoundError:
    print("Error: The sample dataset 'california_housing_train.csv' was not found.")
    print("Please check the path or ensure the file exists.")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

# Next, we can identify common data cleaning tasks:
# 1. Handling missing values
# 2. Correcting data types
# 3. Removing duplicates
# 4. Handling outliers
# 5. Renaming columns

# Let's check for missing values as a first cleaning step:
print("\nChecking for missing values:")
print(df.isnull().sum())


### Checking for Duplicate Rows

To identify duplicate rows, `pandas` DataFrames have a `.duplicated()` method. By default, it marks all duplicates as `True` except for the first occurrence. You can then sum the `True` values to get a count of duplicate rows.

In [ ]:
# Check for duplicate rows in the DataFrame
duplicate_rows = df.duplicated().sum()

print(f"Number of duplicate rows: {duplicate_rows}")

# If you want to see the actual duplicate rows:
# df[df.duplicated(keep=False)] # keep=False marks all occurrences of duplicates as True

In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


In [ ]:
import pandas as pd

# Assuming the 'zomato.csv' file has been uploaded to your Colab environment
# or you provide a direct URL to the dataset.
# You might need to change the file path if your dataset is named differently or stored elsewhere.

dataset_path = '/content/zomato_dataset.csv' # Update this path if necessary

try:
    zomato_df = pd.read_csv(dataset_path)
    print(f"Dataset '{dataset_path}' loaded successfully. Here are the first 5 rows:")
    print(zomato_df.head())
    print("\nAnd here is some basic information about the dataset:")
    zomato_df.info()
except FileNotFoundError:
    print(f"Error: The dataset '{dataset_path}' was not found.")
    print("Please ensure you have uploaded 'zomato.csv' to your Colab environment or provide the correct path/URL.")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

## Data Cleaning Steps

In [ ]:
# Price per vote (value perception)
zomato_df['Price_per_Vote'] = zomato_df['Prices'] / (zomato_df['Votes'] + 1)

# Popularity score
zomato_df['Popularity'] = zomato_df['Dining Votes'] + zomato_df['Delivery_Votes']

# Rating difference
zomato_df['Rating_Diff'] = zomato_df['Dining_Rating'] - zomato_df['Delivery_Rating']

### 1. Handling Missing Values

First, let's identify how many missing values each column has. Then, we'll decide on a strategy to handle them. For `Dining_Rating` and `Delivery_Rating`, we'll fill `NaN` values with `0`, assuming that a missing rating implies it hasn't been rated or doesn't apply. For `Best_Seller`, which is a categorical column, we'll fill `NaN` with 'Not Available'.

In [ ]:
# Check for missing values before handling
print("Missing values before cleaning:")
print(zomato_df.isnull().sum())

# Fill missing values in 'Dining_Rating' and 'Delivery_Rating' with 0
zomato_df['Dining_Rating'] = zomato_df['Dining_Rating'].fillna(0)
zomato_df['Delivery_Rating'] = zomato_df['Delivery_Rating'].fillna(0)

# Fill missing values in 'Best_Seller' with 'Not Available'
zomato_df['Best_Seller'] = zomato_df['Best_Seller'].fillna('Not Available')

# Verify missing values after cleaning
print("\nMissing values after cleaning:")
print(zomato_df.isnull().sum())

### 2. Checking for Duplicate Rows

Duplicate rows can skew analyses and models. Let's check if there are any exact duplicate rows in our dataset and remove them if found.

In [ ]:
# Check for duplicate rows
duplicate_rows_count = zomato_df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_rows_count}")

if duplicate_rows_count > 0:
    print("Removing duplicate rows...")
    zomato_df.drop_duplicates(inplace=True)
    print(f"Number of rows after removing duplicates: {len(zomato_df)}")
else:
    print("No duplicate rows found.")

# Display the first few rows and info of the cleaned DataFrame
print("\nFirst 5 rows of the cleaned DataFrame:")
print(zomato_df.head())
print("\nInfo of the cleaned DataFrame:")
zomato_df.info()

### 3. Distribution of 'Dining_Rating'

Let's visualize the distribution of the `Dining_Rating` column using a histogram. This will show us the frequency of different rating values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(zomato_df['Dining_Rating'], bins=20, kde=True)
plt.title('Distribution of Dining Rating')
plt.xlabel('Dining Rating')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

### 4. Distribution of 'Delivery_Rating'

Now, let's visualize the distribution of the `Delivery_Rating` column. This will provide insights into how delivery ratings are spread.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(zomato_df['Delivery_Rating'], bins=20, kde=True)
plt.title('Distribution of Delivery Rating')
plt.xlabel('Delivery Rating')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

### 5. Distribution of 'Prices'

Let's visualize the distribution of the 'Prices' column using a histogram. This will give us an idea of the price ranges in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(zomato_df['Prices'], bins=30, kde=True)
plt.title('Distribution of Prices')
plt.xlabel('Prices')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

### 6. Scatter Plot of 'Dining_Rating' vs. 'Prices'

Let's create a scatter plot to explore the relationship between `Dining_Rating` and `Prices`. This can help us understand if higher-rated restaurants tend to have higher or lower prices.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.scatterplot(x='Dining_Rating', y='Prices', data=zomato_df, alpha=0.6)
plt.title('Dining Rating vs. Prices')
plt.xlabel('Dining Rating')
plt.ylabel('Prices')
plt.grid(True, alpha=0.75)
plt.show()

### 7. Average Dining Rating by City

Let's calculate the average `Dining_Rating` for each city to see which cities have higher or lower average ratings.

In [ ]:
import pandas as pd

average_dining_rating_by_city = zomato_df.groupby('City')['Dining_Rating'].mean().reset_index()

print("Average Dining Rating by City:")
display(average_dining_rating_by_city.sort_values(by='Dining_Rating', ascending=False))

### 8. Top 10 Cities by Average Dining Rating

To better visualize which cities have the highest average `Dining_Rating`, let's plot the top 10 cities in a bar chart.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get the top 10 cities by average dining rating
top_10_cities_dining = average_dining_rating_by_city.head(10)

plt.figure(figsize=(12, 7))
sns.barplot(x='City', y='Dining_Rating', data=top_10_cities_dining, palette='viridis')
plt.title('Top 10 Cities by Average Dining Rating')
plt.xlabel('City')
plt.ylabel('Average Dining Rating')
plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()

### 9. Top 10 Restaurants by Average Dining Rating

Let's find the top 10 restaurants based on their average `Dining_Rating`. We will exclude restaurants where the `Dining_Rating` was imputed as 0, as this likely means they haven't been rated for dining.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Filter out restaurants with a 0 Dining_Rating (imputed missing values)
rated_restaurants_df = zomato_df[zomato_df['Dining_Rating'] > 0]

# Calculate the average Dining_Rating for each restaurant
average_restaurant_dining_rating = rated_restaurants_df.groupby('Restaurant_Name')['Dining_Rating'].mean().reset_index()

# Sort by Dining_Rating in descending order and get the top 10
top_10_restaurants = average_restaurant_dining_rating.sort_values(by='Dining_Rating', ascending=False).head(10)

print("Top 10 Restaurants by Average Dining Rating:")
display(top_10_restaurants)

### 10. Visualizing Top 10 Restaurants by Average Dining Rating

To make the comparison clearer, let's visualize these top 10 restaurants using a bar chart.

In [ ]:
plt.figure(figsize=(14, 8))
sns.barplot(x='Dining_Rating', y='Restaurant_Name', data=top_10_restaurants, palette='coolwarm', hue='Restaurant_Name', legend=False)
plt.title('Top 10 Restaurants by Average Dining Rating')
plt.xlabel('Average Dining Rating')
plt.ylabel('Restaurant Name')
plt.grid(axis='x', alpha=0.75)
plt.tight_layout()
plt.show()

### 11. Scatter Plot of 'Delivery_Rating' vs. 'Prices'

Now, let's explore the relationship between `Delivery_Rating` and `Prices` using a scatter plot. This will help us understand if restaurants with higher delivery ratings tend to have different pricing strategies.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.scatterplot(x='Delivery_Rating', y='Prices', data=zomato_df, alpha=0.6)
plt.title('Delivery Rating vs. Prices')
plt.xlabel('Delivery Rating')
plt.ylabel('Prices')
plt.grid(True, alpha=0.75)
plt.show()

## 5. Defining and Analyzing 'Underperforming' Restaurants

### Identifying Restaurants with Low Dining Rating and High Prices

To find restaurants that might be underperforming relative to their price point, we'll look for establishments with a `Dining_Rating` in the bottom 25th percentile (excluding the imputed 0 ratings) and `Prices` in the top 25th percentile.

In [ ]:
import numpy as np

# Filter out restaurants with imputed 0 dining ratings for a meaningful analysis
actual_rated_restaurants = zomato_df[zomato_df['Dining_Rating'] > 0]

# Calculate the 25th percentile for Dining_Rating (excluding 0 ratings)
low_dining_rating_threshold = actual_rated_restaurants['Dining_Rating'].quantile(0.25)

# Calculate the 75th percentile for Prices
high_price_threshold = zomato_df['Prices'].quantile(0.75)

print(f"Low Dining Rating Threshold (bottom 25% of actual ratings): {low_dining_rating_threshold:.2f}")
print(f"High Price Threshold (top 25%): {high_price_threshold:.2f}")

# Identify restaurants that meet the criteria
underperforming_restaurants = zomato_df[
    (zomato_df['Dining_Rating'] > 0) & # Ensure actual rating
    (zomato_df['Dining_Rating'] <= low_dining_rating_threshold) &
    (zomato_df['Prices'] >= high_price_threshold)
]

# Display the top 10 such restaurants, sorted by price (descending) then rating (ascending)
print("\nRestaurants with Low Dining Rating and High Prices (Top 10):")
display(underperforming_restaurants.sort_values(by=['Prices', 'Dining_Rating'], ascending=[False, True]).head(10))

### Distribution of Prices for Underperforming Restaurants

Let's visualize the distribution of `Prices` for the `underperforming_restaurants` DataFrame to see the price range within this group.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(underperforming_restaurants['Prices'], bins=10, kde=True)
plt.title('Distribution of Prices for Underperforming Restaurants')
plt.xlabel('Prices')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

### Percentage of Underperforming Restaurants by City

Now, let's calculate the percentage of underperforming restaurants in each city. This analysis will highlight cities where a higher proportion of restaurants have low dining ratings despite high prices.

In [ ]:
# Calculate total restaurants per city
total_restaurants_per_city = zomato_df.groupby('City')['Restaurant_Name'].nunique().reset_index()
total_restaurants_per_city.rename(columns={'Restaurant_Name': 'Total_Restaurants'}, inplace=True)

# Calculate underperforming restaurants per city
underperforming_per_city = underperforming_restaurants.groupby('City')['Restaurant_Name'].nunique().reset_index()
underperforming_per_city.rename(columns={'Restaurant_Name': 'Underperforming_Restaurants'}, inplace=True)

# Merge the two dataframes
restaurant_performance_by_city = pd.merge(
    total_restaurants_per_city,
    underperforming_per_city,
    on='City',
    how='left'
)

# Fill NaN values (cities with no underperforming restaurants) with 0
restaurant_performance_by_city['Underperforming_Restaurants'].fillna(0, inplace=True)

# Calculate the percentage
restaurant_performance_by_city['Percentage_Underperforming'] = (
    restaurant_performance_by_city['Underperforming_Restaurants'] /
    restaurant_performance_by_city['Total_Restaurants']
) * 100

# Display the results, sorted by percentage
print("Percentage of Underperforming Restaurants by City:")
display(restaurant_performance_by_city.sort_values(by='Percentage_Underperforming', ascending=False))

### Visualizing Percentage of Underperforming Restaurants by City

Let's visualize the `Percentage_Underperforming` by `City` using a bar chart to easily identify cities with a higher proportion of underperforming restaurants.

In [ ]:
# Sort the DataFrame for better visualization
plot_data = restaurant_performance_by_city.sort_values(by='Percentage_Underperforming', ascending=False)

plt.figure(figsize=(14, 8))
sns.barplot(x='City', y='Percentage_Underperforming', data=plot_data, palette='coolwarm', hue='City', legend=False)
plt.title('Percentage of Underperforming Restaurants by City')
plt.xlabel('City')
plt.ylabel('Percentage Underperforming (%)')
plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()

### Average Price Comparison: Underperforming vs. All Restaurants

Let's compare the average price of the identified underperforming restaurants with the average price of all restaurants in the dataset. This will highlight if underperforming restaurants are indeed priced significantly higher.

In [ ]:
average_price_all_restaurants = zomato_df['Prices'].mean()
average_price_underperforming = underperforming_restaurants['Prices'].mean()

print(f"Average price of all restaurants: {average_price_all_restaurants:.2f}")
print(f"Average price of underperforming restaurants: {average_price_underperforming:.2f}")

if average_price_underperforming > average_price_all_restaurants:
    price_diff = average_price_underperforming - average_price_all_restaurants
    print(f"Underperforming restaurants are, on average, {price_diff:.2f} more expensive than all restaurants.")
else:
    print("Underperforming restaurants are, on average, not more expensive than all restaurants (or are cheaper).")

### Visualizing Price Differences with a Box Plot

To further illustrate the price difference, let's use a box plot to compare the distribution of `Prices` for 'All Restaurants' versus 'Underperforming Restaurants'.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a temporary DataFrame for plotting the comparison
all_restaurants_prices = zomato_df[['Prices']].copy()
all_restaurants_prices['Category'] = 'All Restaurants'

underperforming_restaurants_prices = underperforming_restaurants[['Prices']].copy()
underperforming_restaurants_prices['Category'] = 'Underperforming Restaurants'

# Concatenate the two DataFrames
comparison_df = pd.concat([all_restaurants_prices, underperforming_restaurants_prices])

plt.figure(figsize=(10, 6))
sns.boxplot(x='Category', y='Prices', data=comparison_df, palette=['skyblue', 'salmon'], hue='Category', legend=False)
plt.title('Price Distribution: All Restaurants vs. Underperforming Restaurants')
plt.xlabel('Restaurant Category')
plt.ylabel('Prices')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
# Identify the city with the highest percentage of underperforming restaurants
highest_density_city = restaurant_performance_by_city.sort_values(
    by='Percentage_Underperforming', ascending=False
).head(1)

print("City with the highest density of expensive underperforming restaurants:")
display(highest_density_city)

### 12. Identifying Restaurants with Low Dining Rating and High Prices

To find restaurants that might be underperforming relative to their price point, we'll look for establishments with a `Dining_Rating` in the bottom 25th percentile (excluding the imputed 0 ratings) and `Prices` in the top 25th percentile.

In [ ]:
import numpy as np

# Filter out restaurants with imputed 0 dining ratings for a meaningful analysis
actual_rated_restaurants = zomato_df[zomato_df['Dining_Rating'] > 0]

# Calculate the 25th percentile for Dining_Rating (excluding 0 ratings)
low_dining_rating_threshold = actual_rated_restaurants['Dining_Rating'].quantile(0.25)

# Calculate the 75th percentile for Prices
high_price_threshold = zomato_df['Prices'].quantile(0.75)

print(f"Low Dining Rating Threshold (bottom 25% of actual ratings): {low_dining_rating_threshold:.2f}")
print(f"High Price Threshold (top 25%): {high_price_threshold:.2f}")

# Identify restaurants that meet the criteria
underperforming_restaurants = zomato_df[
    (zomato_df['Dining_Rating'] > 0) & # Ensure actual rating
    (zomato_df['Dining_Rating'] <= low_dining_rating_threshold) &
    (zomato_df['Prices'] >= high_price_threshold)
]

# Display the top 10 such restaurants, sorted by price (descending) then rating (ascending)
print("\nRestaurants with Low Dining Rating and High Prices (Top 10):")
display(underperforming_restaurants.sort_values(by=['Prices', 'Dining_Rating'], ascending=[False, True]).head(10))

### 13. Distribution of Prices for Underperforming Restaurants

Let's visualize the distribution of `Prices` for the `underperforming_restaurants` DataFrame to see the price range within this group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(underperforming_restaurants['Prices'], bins=10, kde=True)
plt.title('Distribution of Prices for Underperforming Restaurants')
plt.xlabel('Prices')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

### 14. Percentage of Underperforming Restaurants by City

Now, let's calculate the percentage of underperforming restaurants in each city. This analysis will highlight cities where a higher proportion of restaurants have low dining ratings despite high prices.

In [ ]:
# Calculate total restaurants per city
total_restaurants_per_city = zomato_df.groupby('City')['Restaurant_Name'].nunique().reset_index()
total_restaurants_per_city.rename(columns={'Restaurant_Name': 'Total_Restaurants'}, inplace=True)

# Calculate underperforming restaurants per city
underperforming_per_city = underperforming_restaurants.groupby('City')['Restaurant_Name'].nunique().reset_index()
underperforming_per_city.rename(columns={'Restaurant_Name': 'Underperforming_Restaurants'}, inplace=True)

# Merge the two dataframes
restaurant_performance_by_city = pd.merge(
    total_restaurants_per_city,
    underperforming_per_city,
    on='City',
    how='left'
)

# Fill NaN values (cities with no underperforming restaurants) with 0
restaurant_performance_by_city['Underperforming_Restaurants'].fillna(0, inplace=True)

# Calculate the percentage
restaurant_performance_by_city['Percentage_Underperforming'] = (
    restaurant_performance_by_city['Underperforming_Restaurants'] /
    restaurant_performance_by_city['Total_Restaurants']
) * 100

# Display the results, sorted by percentage
print("Percentage of Underperforming Restaurants by City:")
display(restaurant_performance_by_city.sort_values(by='Percentage_Underperforming', ascending=False))

### 15. Visualizing Percentage of Underperforming Restaurants by City

Let's visualize the `Percentage_Underperforming` by `City` using a bar chart to easily identify cities with a higher proportion of underperforming restaurants.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sort the DataFrame for better visualization
plot_data = restaurant_performance_by_city.sort_values(by='Percentage_Underperforming', ascending=False)

plt.figure(figsize=(14, 8))
sns.barplot(x='City', y='Percentage_Underperforming', data=plot_data, palette='coolwarm', hue='City', legend=False)
plt.title('Percentage of Underperforming Restaurants by City')
plt.xlabel('City')
plt.ylabel('Percentage Underperforming (%)')
plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()

### 16. Average Price Comparison: Underperforming vs. All Restaurants

Let's compare the average price of the identified underperforming restaurants with the average price of all restaurants in the dataset. This will highlight if underperforming restaurants are indeed priced significantly higher.

In [ ]:
average_price_all_restaurants = zomato_df['Prices'].mean()
average_price_underperforming = underperforming_restaurants['Prices'].mean()

print(f"Average price of all restaurants: {average_price_all_restaurants:.2f}")
print(f"Average price of underperforming restaurants: {average_price_underperforming:.2f}")

if average_price_underperforming > average_price_all_restaurants:
    price_diff = average_price_underperforming - average_price_all_restaurants
    print(f"Underperforming restaurants are, on average, {price_diff:.2f} more expensive than all restaurants.")
else:
    print("Underperforming restaurants are, on average, not more expensive than all restaurants (or are cheaper).")

### 17. Visualizing Price Differences with a Box Plot

To further illustrate the price difference, let's use a box plot to compare the distribution of `Prices` for 'All Restaurants' versus 'Underperforming Restaurants'.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a temporary DataFrame for plotting the comparison
all_restaurants_prices = zomato_df[['Prices']].copy()
all_restaurants_prices['Category'] = 'All Restaurants'

underperforming_restaurants_prices = underperforming_restaurants[['Prices']].copy()
underperforming_restaurants_prices['Category'] = 'Underperforming Restaurants'

# Concatenate the two DataFrames
comparison_df = pd.concat([all_restaurants_prices, underperforming_restaurants_prices])

plt.figure(figsize=(10, 6))
sns.boxplot(x='Category', y='Prices', data=comparison_df, palette=['skyblue', 'salmon'], hue='Category', legend=False)
plt.title('Price Distribution: All Restaurants vs. Underperforming Restaurants')
plt.xlabel('Restaurant Category')
plt.ylabel('Prices')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
# Identify the city with the highest percentage of underperforming restaurants
highest_density_city = restaurant_performance_by_city.sort_values(
    by='Percentage_Underperforming', ascending=False
).head(1)

print("City with the highest density of expensive underperforming restaurants:")
display(highest_density_city)

## 6. Predictive Modeling: Classifying Underperforming Restaurants

### 1. Define Target Variable: 'Is_Underperforming'

We'll create a new column `Is_Underperforming` based on our previous definition:
*   `Dining_Rating` (excluding 0s) is less than or equal to the 25th percentile (`low_dining_rating_threshold`).
*   `Prices` are greater than or equal to the 75th percentile (`high_price_threshold`).

In [ ]:
import numpy as np

# Ensure we use the original zomato_df for consistency in defining the target
# We already calculated low_dining_rating_threshold and high_price_threshold
# low_dining_rating_threshold = actual_rated_restaurants['Dining_Rating'].quantile(0.25)
# high_price_threshold = zomato_df['Prices'].quantile(0.75)

# Create the target variable 'Is_Underperforming'
zomato_df['Is_Underperforming'] = (
    (zomato_df['Dining_Rating'] > 0) & # Only consider actually rated restaurants for this condition
    (zomato_df['Dining_Rating'] <= low_dining_rating_threshold) &
    (zomato_df['Prices'] >= high_price_threshold)
).astype(int) # Convert boolean to integer (1 for True, 0 for False)

print(f"Number of Underperforming Restaurants: {zomato_df['Is_Underperforming'].sum()}")
print(f"Percentage of Underperforming Restaurants: {zomato_df['Is_Underperforming'].mean() * 100:.2f}%")

# Display the first few rows with the new column
print("\nDataFrame with 'Is_Underperforming' column:")
display(zomato_df[['Restaurant_Name', 'Dining_Rating', 'Prices', 'Is_Underperforming']].head())

### 1.1 Redefining Target Variable: 'Is_Underperforming_City_Based'

To improve our definition and avoid data leakage, let's redefine 'underperforming' using **relative city-based thresholds**. This means that a restaurant's `Dining_Rating` must be in the bottom 25th percentile *of its city* (excluding 0 ratings) and its `Prices` must be in the top 75th percentile *of its city*.

In [ ]:
# Calculate city-specific low dining rating thresholds
city_dining_thresholds = zomato_df[zomato_df['Dining_Rating'] > 0].groupby('City')['Dining_Rating'].quantile(0.25).reset_index()
city_dining_thresholds.rename(columns={'Dining_Rating': 'City_Low_Dining_Rating_Threshold'}, inplace=True)

# Calculate city-specific high price thresholds
city_price_thresholds = zomato_df.groupby('City')['Prices'].quantile(0.75).reset_index()
city_price_thresholds.rename(columns={'Prices': 'City_High_Price_Threshold'}, inplace=True)

# Merge these thresholds back to the main DataFrame
zomato_df_with_thresholds = pd.merge(zomato_df, city_dining_thresholds, on='City', how='left')
zomato_df_with_thresholds = pd.merge(zomato_df_with_thresholds, city_price_thresholds, on='City', how='left')

# Define 'Is_Underperforming_City_Based' using these city-specific thresholds
zomato_df_with_thresholds['Is_Underperforming_City_Based'] = (
    (zomato_df_with_thresholds['Dining_Rating'] > 0) & # Only consider actually rated restaurants for this condition
    (zomato_df_with_thresholds['Dining_Rating'] <= zomato_df_with_thresholds['City_Low_Dining_Rating_Threshold']) &
    (zomato_df_with_thresholds['Prices'] >= zomato_df_with_thresholds['City_High_Price_Threshold'])
).astype(int)

# Update zomato_df with the new column (and the temporary threshold columns)
zomato_df = zomato_df_with_thresholds.copy()

print(f"Number of Underperforming Restaurants (City-Based): {zomato_df['Is_Underperforming_City_Based'].sum()}")
print(f"Percentage of Underperforming Restaurants (City-Based): {zomato_df['Is_Underperforming_City_Based'].mean() * 100:.2f}%")

# Display the first few rows with the new column
print("\nDataFrame with 'Is_Underperforming_City_Based' column:")
display(zomato_df[['Restaurant_Name', 'City', 'Dining_Rating', 'Prices', 'Is_Underperforming_City_Based']].head())

### 2. Feature Selection and Data Splitting (with City-Based Target)

We will now use the new `Is_Underperforming_City_Based` target variable. The numerical features will remain the same: `Dining_Rating`, `Delivery_Rating`, `Dining Votes`, `Delivery_Votes`, `Votes`, and `Prices`. We'll split the data again into training and testing sets, ensuring stratification for the new target, and then scale the features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define features (X) and the NEW target (y_city_based)
X = zomato_df[['Dining_Rating', 'Delivery_Rating', 'Dining Votes', 'Delivery_Votes', 'Votes', 'Prices']]
y_city_based = zomato_df['Is_Underperforming_City_Based']

# Split the data into training and testing sets using the new target
X_train_city, X_test_city, y_train_city, y_test_city = train_test_split(X, y_city_based, test_size=0.3, random_state=42, stratify=y_city_based)

print(f"X_train_city shape: {X_train_city.shape}")
print(f"X_test_city shape: {X_test_city.shape}")
print(f"y_train_city shape: {y_train_city.shape}")
print(f"y_test_city shape: {y_test_city.shape}")

# Scale numerical features for the new split
scaler_city = StandardScaler()
X_train_scaled_city = scaler_city.fit_transform(X_train_city)
X_test_scaled_city = scaler_city.transform(X_test_city)

print("\nFeatures scaled successfully for city-based model.")

### 3. Model Training and Evaluation (with City-Based Target)

Now, let's train a new `RandomForestClassifier` using the city-based target variable and evaluate its performance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize and train the RandomForestClassifier model with the new target
model_city_based = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model_city_based.fit(X_train_scaled_city, y_train_city)

# Make predictions on the test set
y_pred_city = model_city_based.predict(X_test_scaled_city)

# Evaluate the model
print("\nClassification Report (City-Based Model):")
print(classification_report(y_test_city, y_pred_city))

print("\nConfusion Matrix (City-Based Model):")
cm_city = confusion_matrix(y_test_city, y_pred_city)
sns.heatmap(cm_city, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Not Underperforming (0)', 'Underperforming (1)'],
            yticklabels=['Not Underperforming (0)', 'Underperforming (1)'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (City-Based Model)')
plt.show()

accuracy_city = accuracy_score(y_test_city, y_pred_city)
print(f"\nAccuracy (City-Based Model): {accuracy_city:.4f}")

### 4. Feature Importance Analysis (City-Based Model)

Finally, let's analyze the feature importances from our newly trained `RandomForestClassifier` with the city-based target. This will show us which features are most influential in predicting 'Underperforming' restaurants in a more meaningful context.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Get feature importances from the city-based model
feature_importances_city = model_city_based.feature_importances_

# Get feature names from the original DataFrame X
feature_names_city = X.columns # Features are the same as before

# Create a DataFrame for better visualization
importance_df_city = pd.DataFrame({'Feature': feature_names_city, 'Importance': feature_importances_city})
importance_df_city = importance_df_city.sort_values(by='Importance', ascending=False)

print("Feature Importances (City-Based Model):")
display(importance_df_city)

# Visualize feature importances
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df_city, palette='viridis', hue='Feature', legend=False)
plt.title('Feature Importance for Predicting Underperforming Restaurants (City-Based Model)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.grid(axis='x', alpha=0.75)
plt.tight_layout()
plt.show()

### Exporting Model Performance Metrics to CSV

To allow for easy external analysis and record-keeping, we will export the detailed classification report and the confusion matrix to CSV files.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

# --- Export Classification Report ---

# Get the classification report as a dictionary
report_dict = classification_report(y_test_city, y_pred_city, output_dict=True)

# Convert the dictionary to a pandas DataFrame
classification_report_df = pd.DataFrame(report_dict).transpose()

# Save the classification report to a CSV file
classification_report_filename = 'city_based_model_classification_report.csv'
classification_report_df.to_csv(classification_report_filename)
print(f"Classification report saved to {classification_report_filename}")

# --- Export Confusion Matrix ---

# Get the confusion matrix
cm_city_array = confusion_matrix(y_test_city, y_pred_city)

# Convert the confusion matrix to a pandas DataFrame for better labeling
confusion_matrix_df = pd.DataFrame(
    cm_city_array,
    index=['Actual Not Underperforming (0)', 'Actual Underperforming (1)'] ,
    columns=['Predicted Not Underperforming (0)', 'Predicted Underperforming (1)']
)

# Save the confusion matrix to a CSV file
confusion_matrix_filename = 'city_based_model_confusion_matrix.csv'
confusion_matrix_df.to_csv(confusion_matrix_filename)
print(f"Confusion matrix saved to {confusion_matrix_filename}")

print("\nModel performance metrics exported successfully.")

## Predictive Modeling: Classifying Underperforming Restaurants

### 1. Define Target Variable: 'Is_Underperforming'

We'll create a new column `Is_Underperforming` based on our previous definition:
*   `Dining_Rating` (excluding 0s) is less than or equal to the 25th percentile (`low_dining_rating_threshold`).
*   `Prices` are greater than or equal to the 75th percentile (`high_price_threshold`).

In [ ]:
import numpy as np

# Ensure we use the original zomato_df for consistency in defining the target
# We already calculated low_dining_rating_threshold and high_price_threshold
# low_dining_rating_threshold = actual_rated_restaurants['Dining_Rating'].quantile(0.25)
# high_price_threshold = zomato_df['Prices'].quantile(0.75)

# Create the target variable 'Is_Underperforming'
zomato_df['Is_Underperforming'] = (
    (zomato_df['Dining_Rating'] > 0) & # Only consider actually rated restaurants for this condition
    (zomato_df['Dining_Rating'] <= low_dining_rating_threshold) &
    (zomato_df['Prices'] >= high_price_threshold)
).astype(int) # Convert boolean to integer (1 for True, 0 for False)

print(f"Number of Underperforming Restaurants: {zomato_df['Is_Underperforming'].sum()}")
print(f"Percentage of Underperforming Restaurants: {zomato_df['Is_Underperforming'].mean() * 100:.2f}%")

# Display the first few rows with the new column
print("\nDataFrame with 'Is_Underperforming' column:")
display(zomato_df[['Restaurant_Name', 'Dining_Rating', 'Prices', 'Is_Underperforming']].head())

### 1.1 Redefining Target Variable: 'Is_Underperforming_City_Based'

To improve our definition and avoid data leakage, let's redefine 'underperforming' using **relative city-based thresholds**. This means that a restaurant's `Dining_Rating` must be in the bottom 25th percentile *of its city* (excluding 0 ratings) and its `Prices` must be in the top 75th percentile *of its city*.

In [ ]:
# Calculate city-specific low dining rating thresholds
city_dining_thresholds = zomato_df[zomato_df['Dining_Rating'] > 0].groupby('City')['Dining_Rating'].quantile(0.25).reset_index()
city_dining_thresholds.rename(columns={'Dining_Rating': 'City_Low_Dining_Rating_Threshold'}, inplace=True)

# Calculate city-specific high price thresholds
city_price_thresholds = zomato_df.groupby('City')['Prices'].quantile(0.75).reset_index()
city_price_thresholds.rename(columns={'Prices': 'City_High_Price_Threshold'}, inplace=True)

# Merge these thresholds back to the main DataFrame
zomato_df_with_thresholds = pd.merge(zomato_df, city_dining_thresholds, on='City', how='left')
zomato_df_with_thresholds = pd.merge(zomato_df_with_thresholds, city_price_thresholds, on='City', how='left')

# Define 'Is_Underperforming_City_Based' using these city-specific thresholds
zomato_df_with_thresholds['Is_Underperforming_City_Based'] = (
    (zomato_df_with_thresholds['Dining_Rating'] > 0) & # Only consider actually rated restaurants for this condition
    (zomato_df_with_thresholds['Dining_Rating'] <= zomato_df_with_thresholds['City_Low_Dining_Rating_Threshold']) &
    (zomato_df_with_thresholds['Prices'] >= zomato_df_with_thresholds['City_High_Price_Threshold'])
).astype(int)

# Update zomato_df with the new column (and the temporary threshold columns)
zomato_df = zomato_df_with_thresholds.copy()

print(f"Number of Underperforming Restaurants (City-Based): {zomato_df['Is_Underperforming_City_Based'].sum()}")
print(f"Percentage of Underperforming Restaurants (City-Based): {zomato_df['Is_Underperforming_City_Based'].mean() * 100:.2f}%")

# Display the first few rows with the new column
print("\nDataFrame with 'Is_Underperforming_City_Based' column:")
display(zomato_df[['Restaurant_Name', 'City', 'Dining_Rating', 'Prices', 'Is_Underperforming_City_Based']].head())

### 2. Feature Selection and Data Splitting (with City-Based Target)

We will now use the new `Is_Underperforming_City_Based` target variable. The numerical features will remain the same: `Dining_Rating`, `Delivery_Rating`, `Dining Votes`, `Delivery_Votes`, `Votes`, and `Prices`. We'll split the data again into training and testing sets, ensuring stratification for the new target, and then scale the features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define features (X) and the NEW target (y_city_based)
X = zomato_df[['Dining_Rating', 'Delivery_Rating', 'Dining Votes', 'Delivery_Votes', 'Votes', 'Prices']]
y_city_based = zomato_df['Is_Underperforming_City_Based']

# Split the data into training and testing sets using the new target
X_train_city, X_test_city, y_train_city, y_test_city = train_test_split(X, y_city_based, test_size=0.3, random_state=42, stratify=y_city_based)

print(f"X_train_city shape: {X_train_city.shape}")
print(f"X_test_city shape: {X_test_city.shape}")
print(f"y_train_city shape: {y_train_city.shape}")
print(f"y_test_city shape: {y_test_city.shape}")

# Scale numerical features for the new split
scaler_city = StandardScaler()
X_train_scaled_city = scaler_city.fit_transform(X_train_city)
X_test_scaled_city = scaler_city.transform(X_test_city)

print("\nFeatures scaled successfully for city-based model.")

### 3. Model Training and Evaluation (with City-Based Target)

Now, let's train a new `RandomForestClassifier` using the city-based target variable and evaluate its performance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize and train the RandomForestClassifier model with the new target
model_city_based = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model_city_based.fit(X_train_scaled_city, y_train_city)

# Make predictions on the test set
y_pred_city = model_city_based.predict(X_test_scaled_city)

# Evaluate the model
print("\nClassification Report (City-Based Model):")
print(classification_report(y_test_city, y_pred_city))

print("\nConfusion Matrix (City-Based Model):")
cm_city = confusion_matrix(y_test_city, y_pred_city)
sns.heatmap(cm_city, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Not Underperforming (0)', 'Underperforming (1)'],
            yticklabels=['Not Underperforming (0)', 'Underperforming (1)'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (City-Based Model)')
plt.show()

accuracy_city = accuracy_score(y_test_city, y_pred_city)
print(f"\nAccuracy (City-Based Model): {accuracy_city:.4f}")

### 4. Feature Importance Analysis (City-Based Model)

Finally, let's analyze the feature importances from our newly trained `RandomForestClassifier` with the city-based target. This will show us which features are most influential in predicting 'Underperforming' restaurants in a more meaningful context.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Get feature importances from the city-based model
feature_importances_city = model_city_based.feature_importances_

# Get feature names from the original DataFrame X
feature_names_city = X.columns # Features are the same as before

# Create a DataFrame for better visualization
importance_df_city = pd.DataFrame({'Feature': feature_names_city, 'Importance': feature_importances_city})
importance_df_city = importance_df_city.sort_values(by='Importance', ascending=False)

print("Feature Importances (City-Based Model):")
display(importance_df_city)

# Visualize feature importances
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df_city, palette='viridis', hue='Feature', legend=False)
plt.title('Feature Importance for Predicting Underperforming Restaurants (City-Based Model)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.grid(axis='x', alpha=0.75)
plt.tight_layout()
plt.show()

### Exporting Model Performance Metrics to CSV

To allow for easy external analysis and record-keeping, we will export the detailed classification report and the confusion matrix to CSV files.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

# --- Export Classification Report ---

# Get the classification report as a dictionary
report_dict = classification_report(y_test_city, y_pred_city, output_dict=True)

# Convert the dictionary to a pandas DataFrame
classification_report_df = pd.DataFrame(report_dict).transpose()

# Save the classification report to a CSV file
classification_report_filename = 'city_based_model_classification_report.csv'
classification_report_df.to_csv(classification_report_filename)
print(f"Classification report saved to {classification_report_filename}")

# --- Export Confusion Matrix ---

# Get the confusion matrix
cm_city_array = confusion_matrix(y_test_city, y_pred_city)

# Convert the confusion matrix to a pandas DataFrame for better labeling
confusion_matrix_df = pd.DataFrame(
    cm_city_array,
    index=['Actual Not Underperforming (0)', 'Actual Underperforming (1)'],
    columns=['Predicted Not Underperforming (0)', 'Predicted Underperforming (1)']
)

# Save the confusion matrix to a CSV file
confusion_matrix_filename = 'city_based_model_confusion_matrix.csv'
confusion_matrix_df.to_csv(confusion_matrix_filename)
print(f"Confusion matrix saved to {confusion_matrix_filename}")

print("\nModel performance metrics exported successfully.")

### Exporting Key Analysis Results to CSV

To provide a 'full report' in a structured CSV format, I will export several key DataFrames that represent the main insights from our exploratory data analysis and underperforming restaurant identification. This includes:

1.  **Average Dining Rating by City**: Summarizes the mean dining rating for each city.
2.  **Top 10 Restaurants by Dining Rating**: Lists the top restaurants based on their average dining rating.
3.  **Percentage of Underperforming Restaurants by City**: Shows the proportion of underperforming restaurants in each city based on the city-based definition.
4.  **Feature Importances**: Details which features were most influential in predicting underperforming restaurants using the city-based model.
5.  **Identified Underperforming Restaurants**: A list of the restaurants that were classified as underperforming according to our city-based criteria.

In [ ]:
# 1. Export Average Dining Rating by City
average_dining_rating_by_city.to_csv('average_dining_rating_by_city.csv', index=False)
print("Exported 'average_dining_rating_by_city.csv'")

# 2. Export Top 10 Restaurants by Average Dining Rating
top_10_restaurants.to_csv('top_10_restaurants_by_dining_rating.csv', index=False)
print("Exported 'top_10_restaurants_by_dining_rating.csv'")

# 3. Export Percentage of Underperforming Restaurants by City
restaurant_performance_by_city.to_csv('underperforming_restaurants_percentage_by_city.csv', index=False)
print("Exported 'underperforming_restaurants_percentage_by_city.csv'")

# 4. Export Feature Importances (City-Based Model)
importance_df_city.to_csv('feature_importances_city_based_model.csv', index=False)
print("Exported 'feature_importances_city_based_model.csv'")

# 5. Export Identified Underperforming Restaurants
# For simplicity, let's export a subset of key columns for underperforming restaurants, or the whole dataframe if manageable.
underperforming_restaurants.to_csv('identified_underperforming_restaurants.csv', index=False)
print("Exported 'identified_underperforming_restaurants.csv'\n")

print("All requested analysis results have been exported to CSV files.")

### Consolidating All Analysis Results into a Single CSV

To provide a single consolidated report, I will append all the individual analysis results (classification report, confusion matrix, various summary DataFrames) into one CSV file. Each section will be introduced by a descriptive header for clarity.

In [ ]:
import os

output_filename = 'full_analysis_report.csv'

# Initialize the file or clear previous content if it exists
if os.path.exists(output_filename):
    os.remove(output_filename)

def append_df_to_csv(df, title, filename):
    with open(filename, 'a') as f:
        f.write(f'\n# {title}\n') # Add a descriptive header
        # Use index=True for dataframes like classification_report_df and confusion_matrix_df which have meaningful row labels
        # Otherwise, use index=False
        if 'classification_report' in title.lower() or 'confusion_matrix' in title.lower():
            df.to_csv(f, index=True, header=True)
        else:
            df.to_csv(f, index=False, header=True)
        f.write('\n') # Add an extra newline for separation

# Append each relevant DataFrame to the single CSV file
append_df_to_csv(classification_report_df, 'City-Based Model Classification Report', output_filename)
append_df_to_csv(confusion_matrix_df, 'City-Based Model Confusion Matrix', output_filename)
append_df_to_csv(average_dining_rating_by_city, 'Average Dining Rating by City', output_filename)
append_df_to_csv(top_10_restaurants, 'Top 10 Restaurants by Average Dining Rating', output_filename)
append_df_to_csv(restaurant_performance_by_city, 'Percentage of Underperforming Restaurants by City', output_filename)
append_df_to_csv(importance_df_city, 'Feature Importances (City-Based Model)', output_filename)
append_df_to_csv(underperforming_restaurants, 'Identified Underperforming Restaurants', output_filename)

print(f"All analysis results have been combined into a single file: '{output_filename}'.")

In [ ]:
# Price per vote (value perception)
zomato_df['Price_per_Vote'] = zomato_df['Prices'] / (zomato_df['Votes'] + 1)

# Popularity score
zomato_df['Popularity'] = zomato_df['Dining Votes'] + zomato_df['Delivery_Votes']

# Rating difference
zomato_df['Rating_Diff'] = zomato_df['Dining_Rating'] - zomato_df['Delivery_Rating']

print("New features 'Price_per_Vote', 'Popularity', and 'Rating_Diff' have been added to zomato_df.")
print("Displaying the first 5 rows with the new features:")
display(zomato_df[['Restaurant_Name', 'Prices', 'Votes', 'Dining_Rating', 'Delivery_Rating', 'Price_per_Vote', 'Popularity', 'Rating_Diff']].head())

### 2. Feature Selection and Data Splitting

We will use numerical features that are likely to influence whether a restaurant is underperforming. For this initial model, we'll focus on `Dining_Rating`, `Delivery_Rating`, `Dining Votes`, `Delivery_Votes`, `Votes`, and `Prices`. We will then split the data into training and testing sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define features (X) and target (y)
X = zomato_df[['Dining_Rating', 'Delivery_Rating', 'Dining Votes', 'Delivery_Votes', 'Votes', 'Prices']]
y = zomato_df['Is_Underperforming']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Scale numerical features (important for many models, though less critical for Tree-based models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures scaled successfully.")

### 3. Model Training and Evaluation

We'll use a `RandomForestClassifier` for this task, as it's robust and generally performs well on various types of data. After training, we'll evaluate its performance using a classification report and confusion matrix.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize and train the RandomForestClassifier model
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced') # Use class_weight to handle imbalance
model.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test_scaled)

# Evaluate the model
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Not Underperforming (0)', 'Underperforming (1)'],
            yticklabels=['Not Underperforming (0)', 'Underperforming (1)'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")

### 4. Feature Importance Analysis

Let's analyze the feature importances from our trained `RandomForestClassifier`. This will tell us which features were most influential in predicting whether a restaurant is 'Underperforming'.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Get feature importances from the model
feature_importances = model.feature_importances_

# Get feature names from the original DataFrame X
feature_names = X.columns

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

print("Feature Importances:")
display(importance_df)

# Visualize feature importances
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis', hue='Feature', legend=False)
plt.title('Feature Importance for Predicting Underperforming Restaurants')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.grid(axis='x', alpha=0.75)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Predict probabilities for the test set using the city-based model
probabilities_city = model_city_based.predict_proba(X_test_scaled_city)

print("Predicted probabilities (first 5 samples) for each class (0: Not Underperforming, 1: Underperforming):")
display(pd.DataFrame(probabilities_city[:5], columns=['Prob_Not_Underperforming', 'Prob_Underperforming']))

The output above shows the predicted probabilities for the first 5 samples in the test set, using the `model_city_based`. Each row corresponds to a restaurant, and the two columns represent:

*   **Prob_Not_Underperforming (0):** The probability that the restaurant is *not* underperforming.
*   **Prob_Underperforming (1):** The probability that the restaurant *is* underperforming.

For example, if a row shows `[0.98, 0.02]`, it means the model predicts a 98% chance that the restaurant is not underperforming and a 2% chance that it is underperforming. Conversely, `[0.10, 0.90]` would indicate a high probability of being an underperforming restaurant.

# Full Report: Zomato Restaurant Performance Analysis

## 1. Introduction

This report details an analysis of Zomato restaurant data, focusing on identifying factors contributing to restaurant performance and developing a predictive model for 'underperforming' establishments. The goal is to derive insights that can help in understanding restaurant dynamics, pricing strategies, and potential areas for business improvement.

## 2. Data Loading and Initial Inspection

The `zomato_dataset.csv` file was loaded into a pandas DataFrame. The dataset initially contained `123,657` entries across `11` columns, including `Restaurant_Name`, `Dining_Rating`, `Delivery_Rating`, `Prices`, `Cuisine`, `City`, and others. Initial inspection revealed the presence of missing values in `Dining_Rating`, `Delivery_Rating`, and `Best_Seller` columns.

## 3. Data Cleaning

Data cleaning was performed to ensure data quality and prepare for analysis:

*   **Missing Values:**
    *   `Dining_Rating` (initially `32,236` missing) and `Delivery_Rating` (initially `1,280` missing) `NaN` values were imputed with `0`. This assumes that a missing rating implies no rating or non-applicability, which is a practical decision given the data context.
    *   `Best_Seller` (initially `95,715` missing) `NaN` values were filled with 'Not Available', as it is a categorical column.
*   **Duplicate Rows:** The dataset was found to contain `61,452` duplicate rows. These duplicates were removed, reducing the DataFrame size from `123,657` to `62,205` unique entries. This step is crucial for preventing biased analysis and model training.

## 4. Exploratory Data Analysis (EDA)

### 4.1 Rating and Price Distributions

*   **Dining_Rating:** The distribution showed a significant number of 0 ratings (due to imputation), and for rated restaurants, the distribution appeared to be skewed towards higher ratings. (Refer to **Distribution of Dining Rating** histogram).
*   **Delivery_Rating:** Similar to dining ratings, there was a concentration at 0 due to imputation, with rated restaurants showing a tendency towards higher delivery ratings. (Refer to **Distribution of Delivery Rating** histogram).
*   **Prices:** The price distribution was right-skewed, indicating that most restaurants have lower prices, with fewer establishments offering very high-priced items. (Refer to **Distribution of Prices** histogram).

### 4.2 Rating vs. Prices Relationships

*   **Dining_Rating vs. Prices:** A scatter plot revealed no strong linear correlation between `Dining_Rating` and `Prices`. Restaurants across all rating levels (from low to high) exhibited a wide range of prices. (Refer to **Dining Rating vs. Prices** scatter plot).
*   **Delivery_Rating vs. Prices:** Similar to dining ratings, there was no clear linear relationship observed between `Delivery_Rating` and `Prices`. High-rated delivery services were available across various price points. (Refer to **Delivery Rating vs. Prices** scatter plot).

### 4.3 City and Restaurant Performance

*   **Average Dining Rating by City:** Malleshwaram showed the highest average dining rating. The average dining ratings varied across cities, suggesting regional differences in customer satisfaction or rating behaviors. (Refer to **Average Dining Rating by City** table and **Top 10 Cities by Average Dining Rating** bar chart).
*   **Top 10 Restaurants by Average Dining Rating:** Restaurants like 'Toscano', 'Thali and More', and 'AB's - Absolute Barbecues' consistently ranked high, with average dining ratings of 4.7 out of 5 (excluding imputed 0 ratings). These establishments likely represent strong customer satisfaction in their dining experience. (Refer to **Top 10 Restaurants by Average Dining Rating** table and bar chart).

## 5. Defining and Analyzing 'Underperforming' Restaurants

### 5.1 Initial Definition and Limitations

Initially, 'underperforming' restaurants were defined globally as those with a `Dining_Rating` in the bottom 25th percentile (of actual ratings, i.e., <= 3.60) and `Prices` in the top 25th percentile (globally, i.e., >= 299.00). While this identified `3,184` restaurants (5.12%), this definition suffered from potential data leakage because the thresholds were derived from the entire dataset, and the model would learn these global rules directly.

### 5.2 Improved City-Based Definition

To address data leakage and incorporate local market context, the definition of 'underperforming' was refined using **relative city-based thresholds**. A restaurant is now classified as 'underperforming' if:

*   Its `Dining_Rating` (excluding 0s) is less than or equal to the 25th percentile of `Dining_Rating` *within its own city*.
*   Its `Prices` are greater than or equal to the 75th percentile of `Prices` *within its own city*.

Using this improved definition, `3,272` restaurants (`5.26%` of the dataset) were identified as 'underperforming'. This more nuanced approach provides a better representation of underperformance relative to local competition and pricing.

### 5.3 Price Comparison of Underperforming Restaurants

Analysis showed a significant price disparity:

*   **Average Price of All Restaurants:** `243.54`
*   **Average Price of Underperforming Restaurants (City-Based):** `498.53`

On average, underperforming restaurants are approximately `255.00` more expensive than the overall restaurant population. This finding, reinforced by a box plot comparison, highlights that a high price point combined with relatively low dining ratings (within their city context) is a defining characteristic of these establishments.

## 6. Predictive Modeling

### 6.1 Model Goal and Setup

A `RandomForestClassifier` was used to predict whether a restaurant is 'underperforming'. The features used were `Dining_Rating`, `Delivery_Rating`, `Dining Votes`, `Delivery_Votes`, `Votes`, and `Prices`. The data was split into training and testing sets (70/30 split) with stratification to maintain the class distribution, and features were scaled using `StandardScaler`.

### 6.2 Initial Model Performance and Data Leakage

The initial model, trained with the global definition of 'underperforming', showed an accuracy of `1.00`. While seemingly perfect, this indicated significant data leakage, as the target variable was directly derived from the features used for prediction. The model was essentially re-learning its own definition rather than discovering underlying patterns. **It is crucial to re-train the model with the new `Is_Underperforming_City_Based` target to get a valid, non-leaky predictive model.**

### 6.3 Feature Importance (from initial model)

An analysis of feature importances from the *initial, leaky model* confirmed that `Prices` (0.503) and `Dining_Rating` (0.375) were overwhelmingly the most important features. This is expected, as these two features directly formed the basis of the 'underperforming' definition. Other features like `Dining Votes`, `Delivery_Rating`, `Votes`, and `Delivery_Votes` had much lower importance. This further underscores the need to use the `Is_Underperforming_City_Based` target for more meaningful predictions.

## 7. Key Insights and Recommendations

1.  **Price-Performance Mismatch is a Key Indicator:** Restaurants classified as 'underperforming' (based on city-relative low dining ratings and high prices) are, on average, significantly more expensive than other restaurants. This suggests that customers are not perceiving value for money in these establishments.
2.  **Importance of Local Context:** Redefining 'underperforming' using city-based thresholds is critical for accurate analysis and avoids misleading insights from global averages. This makes the concept of 'underperformance' more actionable within specific markets.
3.  **Focus Areas for Underperforming Restaurants:** For restaurants identified as 'underperforming', the primary areas for intervention should be:
    *   **Value Proposition:** Re-evaluate pricing relative to the perceived quality of dining experience. Are customers willing to pay the current prices for the experience offered?
    *   **Service & Quality Improvement:** Investigate the reasons behind low dining ratings. This could involve improving food quality, ambiance, customer service, or overall dining experience.
    *   **Market Analysis:** Understand local competition and pricing within their specific city to adjust strategies effectively.
4.  **Predictive Model Improvement:** While an initial model was built, it's essential to **re-train the `RandomForestClassifier` using the new `Is_Underperforming_City_Based` target variable** to develop a truly predictive model that can identify restaurants at risk of becoming underperforming *before* they fully meet the criteria.

This report provides a foundational understanding of restaurant performance within the Zomato dataset, highlighting key drivers of perceived underperformance and offering a path forward for more robust predictive modeling.